In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import torch
import ast
import re

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, enable_thinking=False)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
input_file = "path/to/input.tsv"
df = pd.read_csv(input_file, sep='\t')

In [ ]:
def translate(commands_list):
    system_content = """
    You will be given a list of sentences in Italian. The sentences represent commands addressed to a robot in a home environment, given by a human. 
    Your task is to translate the following commands from English to Italian.
    
    The commands must be translated according to the following rules:
        1. Translate all the words in the English sentence into Italian, do NOT leave any out.
        2. The main verb must be translated so that it sounds like a direct command, typically using the IMPERATIVE form when appropriate.
        3. Use the meaning of the words appropriate for a HOME CONTEXT.
        4. The translation of the command must NOT mix with translations of other commands.
        5. Do NOT add commas, periods, question marks, or exclamation marks.
        6. KEEP accented letters in the Italian sentence.
        7. SEPARATE words with apostrophes from the following word (i.e. nell' armadio).
    
    You will always receive:
        1. A list containing key-command pairs.
        2. Both the key and the command are strings.
    
    The output must follow these rules:
        1. Your output must be ONLY the list of pairs, in the following format:
            - [("key","translation"), ("key","translation"), ...]
        2. Do NOT include any explanations or reasoning.
        3. Maintain the same order of keys between input and output.

    
    EXAMPLES:
    Input: [("4000_enriched.hrc","go to the bathroom and take the toilet paper"), ("4001_enriched.hrc","that 's a sofa and that 's a lamp"), 
    ("4002_enriched.hrc","take me the jacket from the closet"), ("4003_enriched.hrc","find the bottle in the kitchen and bring it to me"), 
    ("4004_enriched.hrc","move slowly"), ("4005_enriched.hrc","go to the living room and turn on the light"), 
    ("4006_enriched.hrc","take the knife and bring it to the living room")]
    
    Output: [("4000_enriched.hrc","vai in bagno e prendi la carta igienica"), ("4001_enriched.hrc","quello è un divano e quella è una lampada"), 
    ("4002_enriched.hrc","prendimi la giacca nell' armadio"), ("4003_enriched.hrc","trova la bottiglia in cucina e portamela"), 
    ("4004_enriched.hrc","muoviti lentamente"), ("4005_enriched.hrc","vai in salotto e accendi la luce"), 
    ("4006_enriched.hrc","prendi il coltello e portalo in salone")]
    """
    
    prompt = [
      {"role": "system", "content": f"{system_content}"},
      {"role": "user", "content": f"/no_think {commands_list}"}
    ]

    text = tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        do_sample=True,
        temperature=0.2,
        max_new_tokens=512,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
def translate_dataframe(df, batch_size=10, save_interval=10):
    output_file = "path/to/output/qwen2-5.tsv"  # Output file with translations
    
    df_new = df.copy()
    df_new["translations"] = None
    
    rows_to_process = df_new.index.tolist()
    for i in tqdm(range(0, len(rows_to_process), batch_size), desc="Processing: "):
        batch_idx = rows_to_process[i:i+batch_size]
        batch_df = df_new.loc[batch_idx]
        batch_texts = list(batch_df[['id', 'input']].astype(str).itertuples(index=False, name=None))
        
        batch_translations = translate(batch_texts)
        batch_translations = re.sub(r"(\w)'\s*(\w)", r"\1\' \2", batch_translations)

        try:
            translations_list = ast.literal_eval(batch_translations)
            
            for j, idx in enumerate(batch_idx):
                if j < len(translations_list):
                    df_new.at[idx, "translations"] = translations_list[j][-1]

        except Exception as e:
            print(e)
            print(f"{i=}")
            print(batch_translations)
        finally:
            if (i % (batch_size*save_interval)) == 0:
                df_new.to_csv(output_file, sep='\t', index=False)

    df_new.to_csv(output_file, sep='\t', index=False)
    print(f"Translation completed. File saved in: {output_file}")

In [ ]:
translations = translate_dataframe(df, 5)